## ✅ Import Libraries

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

## ✅ STEP 2 — Load Cleaned Dataset and Data Preparation

In [24]:
# Load the cleaned news dataset
df = pd.read_csv("../data/processed/cleaned_news.csv")

df.head()

,content,label,char_length,word_length,clean_content
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,0,5180,889,law enforcement on high alert following threat...
1,Did they post their votes for Hillary already?,0,46,8,did they post their votes for hillary already
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,0,354,52,unbelievable obamas attorney general says most...
3,"Bobby Jindal, raised Hindu, uses story of Chri...",1,8116,1337,bobby jindal raised hindu uses story of christ...
4,SATAN 2: Russia unvelis an image of its terrif...,0,2012,345,satan russia unvelis an image of its terrifyin...


In [25]:
df.isnull().sum()

content           0
label             0
char_length       0
word_length       0
clean_content    48
dtype: int64

In [26]:

df=df[["clean_content", "label"]].dropna()

In [27]:
df.isnull().sum()

clean_content    0
label            0
dtype: int64

In [28]:
df=df.rename(columns={"clean_content": "text"})

## ✅ STEP 3 — Train-Test-Validation Split

In [29]:
# Split the dataset into training, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

## ✅ STEP 4 — Tokenizer + Dataset Conversion

In [30]:
# Prepare the datasets for the transformer model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)


In [31]:
# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding="max_length",
        truncation=True,
        max_length=256
        )

# Apply tokenization to the datasets
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

#rename the label column to labels
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

# Set the format for PyTorch
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/50902 [00:00<?, ? examples/s]

Map:   0%|          | 0/6363 [00:00<?, ? examples/s]

Map:   0%|          | 0/6363 [00:00<?, ? examples/s]

## ✅ STEP 5 — Load DistilBERT

In [32]:
# Load the pre-trained BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
    )

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## STEP 6 — Metrics

In [33]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


## ✅ STEP 7 — Training Setup

In [34]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="models/trained/distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

## 🚀 STEP 8 — Train Model

In [ ]:
# Train the model
trainer.train()

/home/home/.pyenv/versions/3.10.13/envs/Fatocheck/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


## 💾 STEP 9 — Save model

In [ ]:
# Save the trained model and tokenizer
trainer.save_model("models/trained/distilbert")
tokenizer.save_pretrained("models/trained/distilbert")